# Aprendizado de Máquina — Aula prática 07

## Pré-processamento e Pipelines

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Todo notebook deste curso, da Aula 02 em diante, escondeu um `StandardScaler`
dentro de um `Pipeline` e passou adiante com um comentário do tipo *"a Aula 07
explica"*. Chegou a hora.

A regra é uma frase só:

> **toda etapa que aprende alguma coisa dos dados faz parte do modelo — e portanto
> tem de acontecer dentro da dobra.**

O que este notebook acrescenta é o **quanto** custa desobedecer. Porque a resposta
não é "sempre muito": um dos vazamentos que os livros mais advertem é numericamente
inofensivo, e um que quase ninguém menciona fabrica $R^2 = 0{,}40$ a partir de
ruído puro. Vamos medir os dois, mais um terceiro que costuma passar em branco, e
só então montar a maquinaria que impede todos eles de acontecerem.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- dizer quais métodos são sensíveis à escala das covariáveis e **por quê**;
- medir o custo de três vazamentos diferentes e ordená-los por gravidade;
- reconhecer o vazamento por **agrupamento** e corrigi-lo com `GroupKFold`;
- montar um `Pipeline` e explicar por que ele torna o vazamento impossível por
  construção;
- usar `ColumnTransformer` para tratar colunas numéricas e categóricas de formas
  diferentes;
- buscar hiperparâmetros **do pré-processamento** junto com os do modelo;
- limpar um arquivo real que veio quebrado.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são as peças de pré-processamento: o `ColumnTransformer`, que
aplica transformações diferentes a colunas diferentes, o `SimpleImputer` para
faltantes, o `OneHotEncoder` para categóricas, e o `SelectKBest`, que vai servir de
cobaia na Seção 3.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, RobustScaler,
                                   StandardScaler)
from sklearn.tree import DecisionTreeRegressor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. Quem é sensível à escala, e quem não é

"Padronize sempre" é um conselho ruim porque não diz para quem. Vamos descobrir
sozinhos: pegamos um problema, **multiplicamos uma única coluna por 1000** e vemos
quais métodos mudam de resposta.

In [ ]:
rng = np.random.default_rng(0)
n, d = 400, 6


def alvo(M):
    return M[:, 0] + 0.5 * M[:, 1] - M[:, 2] + 0.3 * M[:, 3] * M[:, 4]


X = rng.normal(size=(n, d))
y = alvo(X) + rng.normal(0, 0.5, size=n)
X_te = rng.normal(size=(3000, d))
r_te = alvo(X_te)

# a mesma informacao, com a coluna 0 em outra unidade
escala = np.ones(d)
escala[0] = 1000.0
Xe, Xe_te = X * escala, X_te * escala

metodos = {
    "MQO": skl.LinearRegression(),
    "Ridge (alpha=100)": skl.Ridge(alpha=100.0),
    "arvore": DecisionTreeRegressor(max_depth=6, random_state=0),
    "KNN (k=10)": KNeighborsRegressor(n_neighbors=10),
}

linhas = []
for nome, m in metodos.items():
    a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)
    b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)
    linhas.append({"metodo": nome, "escala original": a, "coluna 0 x1000": b,
                   "mudou?": "sim" if abs(b - a) > 1e-6 * max(a, 1) else "nao"})

# o coeficiente da coluna 0 na Ridge, nas duas escalas (reescalado para comparar)
c1 = skl.Ridge(alpha=100.0).fit(X, y).coef_[0]
c2 = skl.Ridge(alpha=100.0).fit(Xe, y).coef_[0] * 1000
print(f"Ridge: coeficiente da coluna 0 = {c1:.4f} na escala original, "
      f"{c2:.4f} depois de multiplicar a coluna por 1000")
pd.DataFrame(linhas).set_index("metodo").round(4)

Três comportamentos distintos, e cada um por um motivo diferente.

**O MQO não muda nada.** Multiplicar uma coluna por 1000 divide o coeficiente dela
por 1000, e o ajuste é idêntico — a solução de mínimos quadrados é *equivariante*
por reescala. Padronizar antes de um MQO puro não muda predição alguma.

**A árvore também não muda.** Ela só compara valores *dentro* de cada coluna
("$x_j \le t$?"), e a ordem dos valores é a mesma depois de multiplicar por 1000.
Toda a família de árvores herda essa indiferença.

**A Ridge muda.** A penalidade $\lambda\|\beta\|^2$ soma coeficientes de colunas
diferentes, e para isso eles precisam estar em unidades comparáveis. Olhe o
coeficiente impresso acima: na escala original ele é encolhido de $1{,}0$ para
$0{,}79$ pela penalidade; com a coluna mil vezes maior, o coeficiente correspondente
é mil vezes menor, quase não entra na soma $\|\beta\|^2$ — e escapa quase intacto,
em $0{,}99$. **A mesma coluna, a mesma informação, e uma regularização
completamente diferente**, só porque alguém trocou metros por milímetros.

**O KNN muda muito.** A distância euclidiana soma as coordenadas ao quadrado; uma
coluna mil vezes maior domina sozinha e as outras cinco deixam de existir.

A regra, então: **padronize sempre que o método somar coisas de colunas
diferentes** — em uma penalidade (Ridge, Lasso), em uma distância (KNN, SVM,
$k$-médias) ou em uma projeção (PCA).

> **Sua vez.** Repita a tabela usando `Lasso(alpha=0.1)` e `RandomForestRegressor`.
> Antes de rodar, preveja em qual dos dois a reescala vai importar — e explique a
> previsão pelo mecanismo, não pelo resultado.

---
## 3. Vazamento grave: escolher variáveis olhando tudo

Agora o experimento central da aula. O cenário é deliberadamente cruel: $n = 60$
observações, $d = 3000$ covariáveis, e uma resposta $y$ que é **ruído puro**,
independente de $X$. O $R^2$ verdadeiro é zero, e qualquer valor positivo é ilusão.

Duas maneiras de fazer a mesma análise — selecionar as 20 covariáveis mais
correlacionadas com $y$ e ajustar um MQO nelas, avaliando por validação cruzada:

- **errado:** selecionar olhando o conjunto inteiro e *depois* rodar a CV;
- **certo:** pôr a seleção dentro do `Pipeline`, para que ela seja refeita em cada
  dobra usando só o treino daquela dobra.

In [ ]:
# As duas medicoes saem do MESMO laco, com a mesma sequencia de numeros
# aleatorios que gerou a figura desta aula nas notas.
rng_v = np.random.default_rng(21)
n_v, d_v, k_v, n_rep = 60, 3000, 20, 40
dobras = skm.KFold(5, shuffle=True, random_state=0)
unidades = np.array([1, 50, 0.01, 5, 1, 200, 0.1, 2])
beta = np.r_[1.5, 0.02, 80, 0.3, -1.0, 0.005, 10, 0.4]

r2_errado, r2_certo = [], []
r2_escala_errada, r2_escala_certa = [], []
for _ in range(n_rep):
    Xv = rng_v.normal(size=(n_v, d_v))
    yv = rng_v.normal(size=n_v)                 # NENHUMA relacao com Xv

    # (1) ERRADO: seleciona olhando TODOS os dados, e so depois valida
    sel = SelectKBest(f_regression, k=k_v).fit(Xv, yv)
    r2_errado.append(skm.cross_val_score(skl.LinearRegression(), sel.transform(Xv),
                                         yv, cv=dobras, scoring="r2").mean())

    # (2) CERTO: a selecao e' uma etapa do pipeline, refeita dentro de cada dobra
    pipe_sel = Pipeline([("sel", SelectKBest(f_regression, k=k_v)),
                         ("mqo", skl.LinearRegression())])
    r2_certo.append(skm.cross_val_score(pipe_sel, Xv, yv, cv=dobras,
                                        scoring="r2").mean())

    # (3) e (4): o mesmo par, com padronizacao no lugar da selecao, num
    #            problema pequeno, com sinal de verdade e escalas dispares
    Xp = rng_v.normal(size=(60, 8)) * unidades
    yp = Xp @ beta + rng_v.normal(0, 1, 60)

    esc = StandardScaler().fit(Xp)              # ERRADO: aprende com tudo
    r2_escala_errada.append(skm.cross_val_score(
        skl.Ridge(alpha=1.0), esc.transform(Xp), yp, cv=dobras, scoring="r2").mean())
    r2_escala_certa.append(skm.cross_val_score(
        Pipeline([("sc", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))]),
        Xp, yp, cv=dobras, scoring="r2").mean())

print(f"R^2 verdadeiro                     :  0.000")
print(f"selecao FORA da dobra (errado)     : {np.mean(r2_errado):+.3f}")
print(f"selecao DENTRO do pipeline (certo) : {np.mean(r2_certo):+.3f}")

> **A lição.** Um $R^2$ de $+0{,}40$ saiu **do nada**. Não há sinal nenhum nesses
> dados, e ainda assim o procedimento errado reporta um modelo que explica 40% da
> variância — um número que ninguém questionaria num relatório.
>
> O mecanismo: a seleção olhou 3000 colunas de ruído e ficou com as 20 que, *por
> acaso*, mais se pareciam com $y$ **naquela amostra**. As dobras da validação
> cruzada vieram depois e já encontraram essas 20 escolhidas — inclusive nas
> observações que deveriam ser "novas". O modelo não aprendeu nada sobre o mundo;
> aprendeu sobre o próprio conjunto de validação.
>
> Feito certo, a CV devolve $R^2$ **negativo** e denuncia corretamente que não há
> sinal: prever pela média seria melhor.

---
## 4. Vazamento leve: padronizar antes de separar

Este é o vazamento que os livros mais advertem, e o laço da seção anterior já
mediu: um problema com sinal de verdade e escalas muito díspares entre colunas —
o cenário mais favorável possível ao alarme.

In [ ]:
print(f"escala FORA da dobra (errado) : R^2 = {np.mean(r2_escala_errada):.4f}")
print(f"escala DENTRO do pipeline     : R^2 = {np.mean(r2_escala_certa):.4f}")
print(f"diferenca                     : {abs(np.mean(r2_escala_errada) - np.mean(r2_escala_certa)):.2e}")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.6, 3.0))
m1 = [np.mean(r2_errado), np.mean(r2_certo)]
ax1.axhline(0, color="black", lw=0.8)
ax1.bar([0, 1], m1, width=0.55, color=["crimson", "green"])
ax1.set_xticks([0, 1]); ax1.set_xticklabels(["selecao\nfora da dobra",
                                             "selecao\ndentro do pipeline"], fontsize=8)
ax1.set_ylabel("R^2 estimado por CV")
ax1.set_title(f"y e' ruido puro (n={n_v}, d={d_v})", fontsize=9)
for xx, vv in zip([0, 1], m1):
    ax1.annotate(f"{vv:+.2f}", xy=(xx, vv), xytext=(0, 6 if vv > 0 else -14),
                 textcoords="offset points", ha="center", fontsize=9)

m2 = [np.mean(r2_escala_errada), np.mean(r2_escala_certa)]
ax2.bar([0, 1], m2, width=0.55, color=["crimson", "green"])
ax2.set_xticks([0, 1]); ax2.set_xticklabels(["escala\nfora da dobra",
                                             "escala\ndentro do pipeline"], fontsize=8)
ax2.set_ylabel("R^2 estimado por CV")
ax2.set_title("padronizacao, com sinal de verdade", fontsize=9)
ax2.set_ylim(min(m2) - 0.02, max(m2) + 0.02)
for xx, vv in zip([0, 1], m2):
    ax2.annotate(f"{vv:.4f}", xy=(xx, vv), xytext=(0, 6), textcoords="offset points",
                 ha="center", fontsize=9)

Iguais até a quarta casa decimal.

Não é que o aviso clássico esteja errado — é que ele não é ali que mora o perigo. A
padronização vaza duas estatísticas por coluna, cada uma calculada sobre dezenas de
observações, e a diferença entre usar as do conjunto todo e as do treino é
numericamente desprezível. Já a seleção de variáveis vaza **a escolha de quais
colunas olhar**, e essa escolha pode ser inteiramente ditada pelo acaso do conjunto
em que ela foi feita.

A moral não é "relaxe com a padronização". É: **corrija assim mesmo, porque custa
zero** — é só pôr no `Pipeline` —, mas guarde a sua vigilância para o item que
importa.

---
## 5. O vazamento que ninguém vê: observações agrupadas

Existe um terceiro tipo, que não aparece em nenhuma etapa de pré-processamento e
por isso escapa de qualquer `Pipeline`: **a mesma unidade aparecer no treino e no
teste**.

É comum e é traiçoeiro. Cinco medições do mesmo paciente, doze compras do mesmo
cliente, trinta fotos do mesmo indivíduo. Um `train_test_split` aleatório espalha
as medições de um mesmo paciente pelos dois lados, e aí o modelo não precisa
aprender nada sobre pacientes em geral: basta reconhecer *aquele* paciente.

Vamos construir o caso: 100 unidades, 8 medições cada, e uma resposta que depende
fortemente de um efeito da unidade.

In [ ]:
rng_g = np.random.default_rng(7)
n_unidades, por_unidade = 100, 8
grupo = np.repeat(np.arange(n_unidades), por_unidade)

efeito_unidade = rng_g.normal(0, 2.0, size=n_unidades)      # o que nao generaliza
Xg = rng_g.normal(size=(len(grupo), 5))
yg = 0.8 * Xg[:, 0] + efeito_unidade[grupo] + rng_g.normal(0, 0.3, size=len(grupo))

modelo_g = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)

ingenua = skm.cross_val_score(modelo_g, np.c_[Xg, grupo], yg,
                              cv=skm.KFold(5, shuffle=True, random_state=0),
                              scoring="r2").mean()
honesta = skm.cross_val_score(modelo_g, np.c_[Xg, grupo], yg,
                              cv=skm.GroupKFold(5), groups=grupo,
                              scoring="r2").mean()
print(f"CV com KFold      (a unidade aparece nos dois lados): R^2 = {ingenua:+.3f}")
print(f"CV com GroupKFold (cada unidade so de um lado)      : R^2 = {honesta:+.3f}")

A primeira estimativa é uma fantasia: o modelo recebeu a coluna `grupo`, decorou o
efeito de cada unidade e reencontrou as mesmas unidades na validação. A segunda
mede o que a pergunta realmente era — *"o que este modelo faz com uma unidade
nova?"* — e a resposta é bem menos animadora.

Repare que **nenhum `Pipeline` teria salvado você aqui**. O vazamento não está em
uma transformação, está em como as dobras foram formadas. A ferramenta certa é o
`GroupKFold` (ou `StratifiedGroupKFold`, `TimeSeriesSplit` para dados temporais), e
a pergunta que a dispara é sempre a mesma: *duas linhas do meu banco podem se
referir à mesma coisa do mundo?*

### A hierarquia que vale memorizar

Com as três medições na mão, dá para ordenar por gravidade:

1. **Grave** — escolher variáveis, hiperparâmetros ou o modelo olhando dados que
   depois serão usados para avaliar. Inflação sem limite: §3 fabricou $R^2=0{,}40$
   de ruído puro.
2. **Grave** — a mesma unidade nos dois lados da divisão, ou informação do futuro
   em dados temporais. §5, e nenhum `Pipeline` protege.
3. **Leve** — padronizar, imputar pela média, codificar categóricas usando o
   conjunto todo. Numericamente inofensivo com $n$ razoável (§4) — mas corrija
   assim mesmo, porque custa zero.

> **Sua vez.** No experimento da Seção 5, remova a coluna `grupo` das covariáveis
> (o modelo não sabe mais quem é quem) e refaça as duas validações cruzadas. A
> distância entre elas some? Pense antes: o efeito da unidade continua nos dados,
> só que agora precisa ser reconhecido pelos valores de $X$.

---
## 6. O `Pipeline`, por dentro

Um `Pipeline` é uma lista de pares `(nome, transformador)` terminada por um
estimador. O que ele garante é uma disciplina de quem chama `fit` em quê:

- `pipe.fit(X_tr, y_tr)` — cada transformador faz `fit_transform` **no treino**, em
  sequência, e o estimador final é ajustado no resultado;
- `pipe.predict(X_te)` — cada transformador apenas `transform`, com os parâmetros
  aprendidos no treino, e o estimador prediz.

O teste nunca participa de nenhum `fit`. Isso não é uma promessa, é uma
consequência da estrutura — e dá para verificar.

In [ ]:
X_tr, X_va, y_tr, y_va = skm.train_test_split(Xe, y, test_size=0.3, random_state=0)

pipe = Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))])
pipe.fit(X_tr, y_tr)

media_aprendida = pipe.named_steps["escala"].mean_
print("media que o scaler aprendeu (3 primeiras colunas):",
      np.round(media_aprendida[:3], 4))
print("media do TREINO                                  :",
      np.round(X_tr.mean(axis=0)[:3], 4))
print("media do conjunto TODO                           :",
      np.round(Xe.mean(axis=0)[:3], 4))
print(f"\nbate com o treino? {np.allclose(media_aprendida, X_tr.mean(axis=0))}")

E o `Pipeline` inteiro é um estimador como qualquer outro: entra em
`cross_val_score`, em `GridSearchCV`, em `BaggingRegressor`. É essa
composicionalidade que torna a coisa prática — a alternativa seria refazer o
pré-processamento à mão dentro de cada dobra, e é aí que os erros nascem.

---
## 7. `ColumnTransformer`: colunas diferentes, tratamentos diferentes

Os conjuntos de dados do repositório são todos numéricos e completos, o que é uma
sorte que você não vai ter sempre. Vamos fabricar um banco sujo — com colunas
categóricas, faltantes e escalas díspares — para montar o fluxo completo.

In [ ]:
rng_s = np.random.default_rng(11)
n_s = 900
cidades = np.array(["Niteroi", "Rio", "Sao Goncalo", "Marica"])

df = pd.DataFrame({
    "idade": rng_s.integers(18, 80, n_s).astype(float),
    "renda": np.round(rng_s.lognormal(8.2, 0.6, n_s), 2),
    "cidade": rng_s.choice(cidades, n_s, p=[0.4, 0.3, 0.2, 0.1]),
    "plano": rng_s.choice(["basico", "pleno", "premium"], n_s),
})
alvo_s = (0.05 * df["idade"] + 0.0008 * df["renda"]
          + df["plano"].map({"basico": 0.0, "pleno": 1.5, "premium": 3.0})
          + rng_s.normal(0, 0.8, n_s))

# faltantes de verdade: 8% da renda e 5% da idade
df.loc[rng_s.random(n_s) < 0.08, "renda"] = np.nan
df.loc[rng_s.random(n_s) < 0.05, "idade"] = np.nan

print(df.dtypes.to_string())
print(f"\nfaltantes por coluna:\n{df.isna().sum().to_string()}")
df.head()

Repare no problema: `SimpleImputer` só serve para as numéricas, `OneHotEncoder` só
para as categóricas, e `StandardScaler` quebraria nas de texto. O
`ColumnTransformer` resolve aplicando cada receita ao seu grupo de colunas.

In [ ]:
numericas = ["idade", "renda"]
categoricas = ["cidade", "plano"]

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])

modelo = Pipeline([("prep", prep), ("ridge", skl.Ridge(alpha=1.0))])

X_tr, X_te2, y_tr2, y_te2 = skm.train_test_split(df, alvo_s, test_size=0.3,
                                                 random_state=0)
modelo.fit(X_tr, y_tr2)
print("colunas depois do pre-processamento:",
      list(modelo.named_steps["prep"].get_feature_names_out()))
print(f"\nEQM no teste: {mean_squared_error(y_te2, modelo.predict(X_te2)):.4f}")
print(f"variancia de y: {y_te2.var():.4f}")

Duas colunas numéricas viraram duas; duas categóricas viraram sete *dummies*. E o
`handle_unknown="ignore"` não é preciosismo: uma categoria pode existir no teste e
não no treino — basta a divisão dar azar com "Maricá", que aparece em 10% das
linhas — e sem ele o `predict` levanta exceção em produção, meses depois, quando
ninguém lembrar por quê.

Note também que o `ColumnTransformer` é ele próprio uma etapa de um `Pipeline`
maior, e que dentro dele há outro `Pipeline` aninhado para as numéricas. A estrutura
é recursiva de propósito: qualquer análise, por mais complicada, cabe em um único
objeto com um `fit` e um `predict`.

---
## 8. Buscar hiperparâmetros do pré-processamento

Se o pré-processamento é parte do modelo, então as escolhas dele — mediana ou
média na imputação? `StandardScaler` ou `RobustScaler`? — são hiperparâmetros como
qualquer outro, e devem ser escolhidas por validação cruzada, não por hábito.

A sintaxe usa `__` para descer na hierarquia: `prep__num__imp__strategy` quer dizer
*"o parâmetro `strategy` da etapa `imp`, dentro da etapa `num`, dentro da etapa
`prep`"*.

In [ ]:
grade = {
    "prep__num__imp__strategy": ["mean", "median", "most_frequent"],
    "prep__num__sc": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    "ridge__alpha": np.logspace(-2, 3, 12),
}

busca = skm.GridSearchCV(modelo, grade, cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error", n_jobs=-1)
busca.fit(X_tr, y_tr2)

print("melhores escolhas:")
for chave, valor in busca.best_params_.items():
    print(f"   {chave:28s} = {valor}")
print(f"\nEQM de CV no vencedor: {-busca.best_score_:.4f}")
print(f"EQM no teste         : {mean_squared_error(y_te2, busca.predict(X_te2)):.4f}")
print(f"combinacoes avaliadas: {len(busca.cv_results_['mean_test_score'])}")

In [ ]:
res = pd.DataFrame(busca.cv_results_)
res["escala"] = res["param_prep__num__sc"].astype(str).str.replace("()", "", regex=False)
resumo = (res.groupby(["escala", "param_prep__num__imp__strategy"])["mean_test_score"]
          .max().unstack() * -1)
resumo.round(4)

A tabela separa duas coisas. **Qual `Scaler`** não muda praticamente nada: as três
linhas são iguais até a terceira casa. **Qual imputação**, muda muito: preencher
faltantes com a *moda* de uma variável contínua é uma ideia ruim, e custa 40% de
erro a mais. A validação cruzada pegou isso sozinha — que é exatamente o argumento
para pôr o pré-processamento na grade em vez de decidi-lo por hábito.

O importante é que a busca aconteceu
**dentro** da validação cruzada: cada combinação foi avaliada refazendo imputação e
escala em cada dobra. Fazer isso à mão, sem `Pipeline`, é onde os erros da Seção 3
nascem.

> **Sua vez.** Acrescente à grade um `SelectKBest(f_regression)` como etapa
> intermediária e busque também o `k`. Confira que o número de combinações se
> multiplica — e cronometre. É essa explosão que motiva o `RandomizedSearchCV`,
> que sorteia combinações em vez de varrer todas.

---
## 9. Um arquivo real, quebrado

Fechamos com um problema que nenhum livro cobre e todo mundo enfrenta: o arquivo
chega errado. O `bank_train_redux.csv` do repositório — que a Aula 09 vai usar — tem
um defeito de exportação, e é instrutivo diagnosticá-lo antes de consertar.

Vamos ler só as primeiras linhas, porque o arquivo tem cerca de 100 MB.

In [ ]:
import os

_nome = "bank_train_redux.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

banco = pd.read_csv(_fonte, nrows=20_000)
print("dimensoes:", banco.shape)
print("\nultimas 3 colunas:", list(banco.columns)[-3:])
print("\ntipos das ultimas 3 colunas:")
print(banco.dtypes.tail(3).to_string())

Dois sintomas. O nome da última coluna é `var_199;;;;;;;` — sobraram separadores no
cabeçalho. E o `dtype` dela é `object`, isto é, texto, enquanto todas as outras 199
são numéricas. Quando uma coluna que deveria ser número vem como texto, quase sempre
há sujeira dentro dos valores.

In [ ]:
coluna = banco.columns[-1]
print("tres valores dessa coluna, como vieram:")
print(banco[coluna].head(3).to_string())
print(f"\ntentando converter direto: ", end="")
try:
    pd.to_numeric(banco[coluna])
    print("funcionou")
except ValueError as erro:
    print(f"ValueError -> {str(erro)[:70]}")

In [ ]:
limpo = banco.copy()
limpo = limpo.replace(to_replace=";", value="", regex=True)
limpo = limpo.rename(columns={coluna: "var_199"})
limpo["var_199"] = pd.to_numeric(limpo["var_199"])

print(f"colunas nao numericas antes : {(banco.dtypes == object).sum()}")
print(f"colunas nao numericas depois: {(limpo.dtypes == object).sum()}"
      f"   (a ID_code, que e' texto de verdade)")
print(f"\nvar_199 agora: media {limpo['var_199'].mean():.3f}, "
      f"desvio {limpo['var_199'].std():.3f}")
print(f"prevalencia da classe positiva: {limpo['target'].mean():.4f}")

Três linhas resolveram. Mas repare no que aconteceria sem esse diagnóstico: a
coluna `var_199` seria descartada em silêncio pelo `ColumnTransformer` (ou viraria
duzentas *dummies* de texto no `OneHotEncoder`), e ninguém notaria.

E note onde essa limpeza **não** pode entrar: ela não depende dos dados de treino,
é uma correção de formato do arquivo. Faz sentido fazê-la uma vez, antes de tudo.
A distinção é essa — o que aprende parâmetros dos dados vai para o `Pipeline`; o
que é conserto determinístico de formato pode acontecer antes.

> **Sua vez.** Monte um `Pipeline` completo para este banco: descarte `ID_code`,
> padronize as 200 colunas `var_*` e ajuste uma `LogisticRegression`. Avalie por
> validação cruzada de 5 dobras. Guarde o resultado — a Aula 09 vai mostrar por que
> a acurácia que você obtiver não quer dizer o que parece.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| sensibilidade à escala | §2 | MQO e árvores não mudam; Ridge e KNN mudam — padronize quem soma colunas |
| vazamento por seleção | §3 | $R^2 = +0{,}40$ a partir de **ruído puro**; feito certo, $-0{,}69$ |
| vazamento por escala | §4 | $0{,}8466$ contra $0{,}8466$: numericamente irrelevante |
| vazamento por agrupamento | §5 | `KFold` mente quando a mesma unidade está nos dois lados; use `GroupKFold` |
| hierarquia | §5 | seleção e agrupamento são graves; escala é leve — mas corrija todos |
| `Pipeline` | §6 | o scaler aprende a média **do treino**; o teste nunca entra num `fit` |
| `ColumnTransformer` | §7 | numéricas e categóricas em ramos separados; `handle_unknown="ignore"` |
| busca no pré-processamento | §8 | `prep__num__imp__strategy` é hiperparâmetro como qualquer outro |
| arquivo quebrado | §9 | `var_199;;;;;;;` viraria lixo silencioso sem o diagnóstico |

**Leitura recomendada.** [AME] §2.1.2 (o papel do pré-processamento na estimação).
[ISLP] os laboratórios dos Capítulos 5 e 6, onde o `Pipeline` aparece pela primeira
vez, e a discussão do Capítulo 6 sobre por que a seleção de variáveis precisa estar
dentro da validação cruzada — que é exatamente a medição da Seção 3. Vale também o
[ESL] §7.10.2, que é a origem desse exemplo.

**Para praticar.** `Lista teorica 07.pdf` (teórica, com gabarito) e
`Lista prática 07.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** Fecha o Bloco I. A Aula 08 abre a classificação: a resposta deixa de
ser um número e passa a ser uma classe, o risco deixa de ser erro quadrático, e
quase tudo o que construímos até aqui precisa ser reescrito — com a vantagem de
que agora sabemos exatamente o que estamos reescrevendo.